In [ ]:
import os
import cv2
import random
import time
import itertools
import optuna
import gc
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

In [ ]:
# Tắt log của optuna và định dạng hiển thị số thập phân của pandas
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# ===================================================================
# 0. SETUP & LOAD DATA
# ===================================================================
# Thiết lập seed ngẫu nhiên, đảm bảo tính tái lập và ổn định của kết quả
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
SEED = 42
seed_everything(SEED)

# Đường dẫn đến 3 thư mục chứa hình ảnh tương ứng với 3 nhãn thời tiết trên Kaggle
DATA_SOURCES = {
    'haze': '/kaggle/input/datasets/jehanbhathena/weather-dataset/dataset/fogsmog',
    'rain': '/kaggle/input/datasets/jehanbhathena/weather-dataset/dataset/rain',
    'shine': '/kaggle/input/datasets/pratik2901/multiclass-weather-dataset/Multi-class Weather Dataset/Shine'
}
# Kích thước ảnh chuẩn hóa và tham số HOG
IMG_SIZE = 256
HOG_PPC = 16

# Duyệt qua từng nhãn và- đường dẫn, thu thập tất cả các ảnh và nhãn tương ứng
print("[0] LOADING PATHS...")
img_paths, labels_raw = [], []
for label, path in DATA_SOURCES.items():
    if os.path.exists(path):
        files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        img_paths.extend(files)
        labels_raw.extend([label] * len(files))

# Mã hóa nhãn từ dạng chuỗi sang số nguyên để mô hình có thể xử lý
le = LabelEncoder()
y_encoded = le.fit_transform(labels_raw)
class_names = le.classes_

# ===================================================================
# 1. FEATURE EXTRACTION
# ===================================================================
# Chuyển ảnh sang không gian màu HSV, tính trung bình (mean) và độ lệch chuẩn (std) của từng kênh
def get_color_stats(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return [np.mean(hsv[:,:,i]) for i in range(3)] + [np.std(hsv[:,:,i]) for i in range(3)]

# Tính toán ma trận Sobel để lấy đặc trưng biên (edges), sau đó tính trung bình và phương sai của độ lớn biên
def get_sobel_stats(gray):
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    return [np.mean(mag), np.var(mag)]

# Tính toán LBP để mô tả kết cấu (texture) cục bộ của bức ảnh
def get_lbp_hist(gray):
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    hist = hist.astype("float"); return hist / (hist.sum() + 1e-7)

# Tính toán HOG để trích xuất hình dạng và đường bao của các vật thể trong ảnh
def get_hog_stats(gray):
    hog_v = hog(gray, orientations=12, pixels_per_cell=(HOG_PPC, HOG_PPC), cells_per_block=(2,2), visualize=False, feature_vector=True)
    return [np.mean(hog_v), np.std(hog_v), np.max(hog_v)]

# Phân tích GLCM để trích xuất các đặc trưng về kết cấu như độ tương phản, tương quan, năng lượng và đồng nhất
def get_glcm_stats(gray):
    glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    return np.concatenate([contrast, correlation, energy, homogeneity])

# Hàm tổng hợp để trích xuất tất cả các đặc trưng đã định nghĩa ở trên cho mỗi ảnh trong danh sách đường dẫn
def extract_all(paths):
    l_c, l_s, l_l, l_h, l_glcm = [], [], [], [], []
    print("[1] Extracting features...")
    for path in tqdm(paths):
        img = cv2.imread(path); img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        l_c.append(get_color_stats(img))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        l_s.append(get_sobel_stats(gray)); l_l.append(get_lbp_hist(gray))
        l_h.append(get_hog_stats(gray)); l_glcm.append(get_glcm_stats(gray))
    return {'Color': np.array(l_c), 'Sobel': np.array(l_s), 'LBP': np.array(l_l), 'HOG': np.array(l_h), 'GLCM': np.array(l_glcm)}

all_features = extract_all(img_paths)

# Chia dữ liệu thành tập huấn luyện (train: 80%) và kiểm tra (test: 20%)
indices = np.arange(len(img_paths))
X_train_idx, X_test_idx, y_train, y_test = train_test_split(indices, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded)

# Thiết lập chiến lược kiểm tra chéo (Cross-Validation) với 5 folds
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Tạo tất cả các tổ hợp có thể của 5 loại đặc trưng để thử nghiệm (tổng cộng 31 tổ hợp)
base_features = ['Color', 'Sobel', 'LBP', 'HOG', 'GLCM']
all_combinations = []
for r in range(1, len(base_features) + 1):
    for combo in itertools.combinations(base_features, r):
        all_combinations.append({'name': " + ".join(combo), 'components': list(combo)})

# Vẽ ma trận nhầm lẫn (confusion matrix) dưới dạng heatmap để trực quan hóa hiệu suất của mô hình trên tập kiểm tra
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=10); plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.tight_layout(); plt.show()

# ===================================================================
# 2. MAIN LOOP: XGBOOST (DEFAULT vs TUNED + WEIGHT DECISION)
# ===================================================================
results = []
checkpoint_file = 'xgb_smart_tuning_results.csv'

# Lưu trạng thái mô hình tốt nhất
best_overall_f1 = 0.0
best_model_info = {}

print("\n" + "="*80)
print(f" STARTING EXPERIMENT")
print(f" 1. Calc Default CV | 2. Tune Params & Weight | 3. Final Test & Latency")
print("="*80)

# Duyệt qua từng tổ hợp đặc trưng và thực hiện 3 giai đoạn
for idx, combo in enumerate(all_combinations):
    # Tên tổ hợp và các thành phần đặc trưng
    combo_name = combo['name']
    comps = combo['components']
    
    # Dữ liệu đặc trưng cho tập Train và Test dựa trên các thành phần đã chọn
    X_train_curr = np.hstack([all_features[c][X_train_idx] for c in comps])
    X_test_curr  = np.hstack([all_features[c][X_test_idx] for c in comps])
    
    print(f"\n{'#'*60}")
    print(f" [{idx+1}/31] COMBO: {combo_name}")
    print(f"{'#'*60}")
    
    # --- PHASE A: CALCULATE DEFAULT CV SCORE (BASELINE) ---
    # Sử dụng XGBoost với tham số mặc định để có điểm chuẩn (baseline) trước khi tiến hành tuning
    xgb_default = XGBClassifier(eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
    
    # Tạo pipeline để chuẩn hóa dữ liệu trước khi đưa vào mô hình
    pipe_default = Pipeline([('scaler', StandardScaler()), ('clf', xgb_default)])
    
    # Đo CV trên tập Train
    scores_default = cross_val_score(pipe_default, X_train_curr, y_train, cv=cv_strategy, scoring='f1_macro', n_jobs=-1)
    default_cv_f1 = scores_default.mean()
    print(f"   [Baseline] Default CV F1: {default_cv_f1:.4f}")

    # --- PHASE B: OPTUNA TUNING (PARAMS + WEIGHT) ---
    def objective(trial):
        # 1. Điều chỉnh siêu tham số của XGBoost
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'eval_metric': 'mlogloss', 'random_state': SEED, 'n_jobs': -1
        }
        
        # 2. Chiến lược Tune Balancing (Có dùng Weight hay không?)
        use_balancing = trial.suggest_categorical('use_balancing', [True, False])
        
        # 3. Vòng lặp CV để đánh giá hiệu suất của mô hình với tham số hiện tại
        cv_scores = []
        for train_idx_cv, val_idx_cv in cv_strategy.split(X_train_curr, y_train):
            # Lấy dữ liệu thô cho mỗi Fold
            X_tr_fold = X_train_curr[train_idx_cv]
            X_val_fold = X_train_curr[val_idx_cv]
            
            y_tr_fold = y_train[train_idx_cv]
            y_val_fold = y_train[val_idx_cv]
            
            sample_w = None
            if use_balancing:
                sample_w = compute_sample_weight('balanced', y_tr_fold)
            
            clf = XGBClassifier(**params)
            clf.fit(X_tr_fold, y_tr_fold, sample_weight=sample_w)
            
            preds = clf.predict(X_val_fold)
            cv_scores.append(f1_score(y_val_fold, preds, average='macro'))
            
        return np.mean(cv_scores)

    study = optuna.create_study(direction='maximize')
    
    # Thêm tham số mặc định vào Enqueue (Để đảm bảo Tuned >= Default)
    study.enqueue_trial({
        'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'subsample': 1.0, 
        'colsample_bytree': 1.0, 'min_child_weight': 1, 'gamma': 0, 'use_balancing': False
    })
    
    # Bắt đầu quá trình tối ưu hóa với Optuna, tìm kiếm tham số tốt nhất dựa trên điểm CV F1
    study.optimize(objective, n_trials=50) 
    
    # Lấy tham số tốt nhất và điểm CV tương ứng sau khi tuning
    best_params = study.best_params
    tuned_cv_f1 = study.best_value
    
    # --- PHASE C: FINAL TRAIN & TEST PREDICT (MEASURE LATENCY) ---
    # Sử dụng bộ siêu tham số tốt nhất (best_params) vừa tìm được
    use_balancing_final = best_params.pop('use_balancing')
    final_xgb = XGBClassifier(**best_params, eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
    
    final_weights = None
    if use_balancing_final:
        final_weights = compute_sample_weight('balanced', y_train)
    
    # Train
    final_xgb.fit(X_train_curr, y_train, sample_weight=final_weights)
    
    # Đo thời gian dự đoán trên tập Test để tính toán độ trễ (latency)
    t0 = time.time()
    y_test_pred = final_xgb.predict(X_test_curr)
    t1 = time.time()
    
    # Tính độ trễ Latency (ms per image)
    latency_ms = ((t1 - t0) * 1000) / len(X_test_curr)
    
    # Tính điểm F1 trên tập Test với mô hình đã được tối ưu hóa
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    # Lưu mô hình tốt nhất
    if test_f1 > best_overall_f1:
        best_overall_f1 = test_f1
        best_model_info = {
            'combination': combo_name,
            'f1_score': test_f1,
            'params': best_params
        }
        
        # Cập nhật file pkl ngay khi tìm thấy model mới tốt hơn
        print(f"[!] Kỷ lục mới! F1: {test_f1:.4f}. Đang lưu mô hình...")
        joblib.dump(final_xgb, '/kaggle/working/best_xgb_model.pkl')
        joblib.dump(le, '/kaggle/working/label_encoder.pkl')
        
        # Lưu lại thông tin bộ đặc trưng tốt nhất để trích xuất
        with open('/kaggle/working/best_features.txt', 'w') as f:
            f.write(combo_name)

    # --- REPORT ---
    print(f"   >>> Result: Def_CV={default_cv_f1:.4f} | Tuned_CV={tuned_cv_f1:.4f} | Test_F1={test_f1:.4f}")
    print(f"   >>> Latency: {latency_ms:.2f} ms/img | Mode: {'Weighted' if use_balancing_final else 'Unweighted'}")
    print("-" * 50)
    print(classification_report(y_test, y_test_pred, target_names=class_names, digits=4))
    
    plot_title = f"XGB | {combo_name}\nMode: {'Weighted' if use_balancing_final else 'Unweighted'} | F1: {test_f1:.4f}"
    plot_cm(y_test, y_test_pred, plot_title)
    print("-" * 50)

    results.append({
        'Combination': combo_name,
        'Default CV F1': round(default_cv_f1, 4), # <--- Đã thêm cột này
        'Tuned CV F1': round(tuned_cv_f1, 4),
        'Test F1 (Final)': round(test_f1, 4),
        'Improvement': round(tuned_cv_f1 - default_cv_f1, 4),
        'Latency (ms/img)': round(latency_ms, 4), # <--- Đã thêm cột này
        'Balancing Mode': 'Weighted' if use_balancing_final else 'Unweighted',
        'Best Params': str(best_params)
    })
    
    pd.DataFrame(results).to_csv(checkpoint_file, index=False)
    gc.collect()

df_final = pd.DataFrame(results)
print("\n>>> RESULTS SUMMARY:")
cols = ['Combination', 'Default CV F1', 'Tuned CV F1', 'Test F1 (Final)', 'Latency (ms/img)']
print(df_final.sort_values(by='Test F1 (Final)', ascending=False)[cols].head(10).to_string(index=False))

print("\n" + "-"*50)
print(f"    BEST MODEL OVERALL:")
print(f"  - Features: {best_model_info['combination']}")
print(f"  - F1 Score: {best_model_info['f1_score']:.4f}")
print("-" * 50)